In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/viagens_teste/bronze/arquivos/"
    )
)

In [0]:
df_a = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/A_Passagens.csv",
    header =True,
    inferSchema=True,
    sep=";"
)
display(df_a)

In [0]:
df_a.printSchema()

df_a.columns

In [0]:
df_b = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/B_Passagens.csv",
    header =True,
    inferSchema=True,
    sep=";"
)
display(df_b)

In [0]:
df_b.printSchema()

df_b.columns

In [0]:
from pyspark.sql.functions import col, lit

In [0]:
df_a_padrao = df_a.select(
    col("Codigo").alias("id_solicitacao"),
    col("Nome Completo").alias("viajante"),
    col("Data Solicitacao").alias("data_solicitacao"),
    col("Data Ida").alias("data_viagem"),
    col("Centro de Custo").alias("centro_custo_origem"),
    col("Origem").alias("origem"),
    col("Destino").alias("destino"),
    col("Valor Total").alias("valor"),
    col("Situacao").alias("situacao")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
display(df_a_padrao)

In [0]:
df_b_padrao = df_b.select(
col("Nr Reserva").alias("id_solicitacao"),
col("Passageiro").alias("viajante"),
col("Criado Em").alias("data_solicitacao"),
col("Embarque").alias("data_viagem"),
col("CC").alias("centro_custo_origem"),
col("De").alias("origem"),
col("Para").alias("destino"),
col("Tarifa").alias("valor"),
col("Status").alias("situacao")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)
    

In [0]:
display(df_b_padrao)

In [0]:
df_a_padrao.columns == df_b_padrao.columns

In [0]:
df_passagens = df_a_padrao.unionByName(df_b_padrao)

display(df_passagens)

In [0]:
display(df_passagens)

In [0]:
df_a.count()

In [0]:
df_a_padrao = df_a_padrao.filter(
    col("id_solicitacao").isNotNull()
)

df_b_padrao = df_b_padrao.filter(
    col("id_solicitacao").isNotNull()
)

In [0]:
df_b_padrao.count()

In [0]:
df_passagens = df_a_padrao.unionByName(df_b_padrao)

In [0]:
print(df_a_padrao.count())
print(df_b_padrao.count())

In [0]:
print(df_passagens.count())

In [0]:
df_passagens.groupBy("agencia").count().show()

In [0]:
df_passagens.select(
    "agencia",
    "valor"
    ).show(20, truncate=False)

df_passagens.printSchema()

In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
df_passagens = df_passagens.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$ ",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_passagens.select(
    "agencia",
    "valor",
    "valor_numerico"
).show(20, truncate=False)

In [0]:
df_passagens.printSchema()

In [0]:
from pyspark.sql.functions import sum

df_passagens.select(
    sum("valor_numerico").alias("total_passagens")
).show()

In [0]:
df_passagens.groupBy("agencia").agg(
    sum("valor_numerico").alias("total_passagens")
).show()

In [0]:
from pyspark.sql.functions import datediff

In [0]:
df_passagens = df_passagens.withColumn(
    "dias_antecedencia",
    datediff(
        col("data_viagem"),
        col("data_solicitacao")
    )
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_solicitacao",
    "data_viagem",
    "dias_antecedencia"
).show(20, truncate=False)

In [0]:
from pyspark.sql.functions import when

In [0]:
df_passagens = df_passagens.withColumn(
    "prazo",
    when(
        col("dias_antecedencia") >= 10,
        "Dentro"
    ).otherwise("Fora")
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_solicitacao",
    "data_viagem",
    "dias_antecedencia",
    "prazo"
).show(20, truncate=False)

In [0]:
df_passagens.groupBy("prazo").count().show()

In [0]:
df_passagens.groupBy("prazo").agg(
    sum("valor_numerico").alias("valor_total")
).show()

In [0]:
df_passagens = df_passagens.withColumn(
    "tipo_despesa",
    lit("Aereo")
)

In [0]:
from pyspark.sql.functions import month

In [0]:
df_passagens = df_passagens.withColumn(
    "trimestre",
    when(
        month(col("data_viagem")).between(1, 3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_viagem")).between(4, 6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_viagem")).between(7, 9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_viagem",
    "tipo_despesa",
    "trimestre"
).show(20, truncate=False)

In [0]:
df_passagens.groupBy("trimestre").count().show()

In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)
display(df_map)

In [0]:
df_map.printSchema()
df_map.columns

In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens = df_passagens.drop(
    "centro_custo",
    "diretoria"
)

In [0]:
df_passagens.printSchema()

In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens.printSchema()

In [0]:
df_passagens.count()

In [0]:
df_passagens.filter(
    col("centro_custo").isNull()
).count()

In [0]:
df_passagens.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.silver.passagens")

In [0]:
df_teste_leitura = spark.table(
    "viagens_teste.silver.passagens"
)

display(df_teste_leitura)